# Predicting Smartphone Addiction - Ultra SOTA Top-100 Rank-01 Multi-Stream Ensemble Pipeline

## Overview
This notebook implements the **Ultra SOTA Top-100 Rank-01 Multi-Stream Ensemble**:
1. **Stream 1 Anchor (In-House SOTA 0.97024 Champion)**: 50-model cross-fitted logit-stack ensemble across 10 stratified folds.
2. **Stream 2 Multi-Scale Engine (50 Models on Seed 2026)**:
   - XGBoost Depth-6 Hist Booster (CUDA)
   - XGBoost Depth-5 Regularized Booster (CUDA)
   - LightGBM NumLeaves=63 Booster (Multi-Threaded CPU)
   - Deep XGBoost Depth-8 Hist Booster (CUDA)
   - CatBoost Native Ordered Target Statistics (GPU)
3. **Engineered Feature Space**:
   - Transductive XGBoost Imputation for uncorrupted composition ratios
   - Decimal Lattice: `frac_col`, `d1_col`, `is_int`, `is_half`
   - Transductive Population Frequencies across 987,671 samples
   - Nested 10-Fold In-Fold Bayesian Target Encodings (`SMOOTH=10.0`)
4. **Cross-Fitted Logit Meta-Stacking & Rank-01 Normalization**:
   - $R(p_i) = \frac{\text{Rank}(p_i) - 1}{N - 1} \in [0, 1]$
   - Convex combination: $R_{\text{final}} = 0.55 \cdot R_{\text{Stream1}} + 0.45 \cdot R_{\text{Stream2}}$

In [ ]:
import os
import sys
import time
import gc
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
print('Environment initialized successfully.')

In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

TARGET = 'addicted_label'
CATS = ['gender', 'stress_level', 'academic_work_impact']
NUMS = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
        'work_study_hours', 'sleep_hours', 'notifications_per_day',
        'app_opens_per_day', 'weekend_screen_time']
ALL_RAW = NUMS + CATS
FRAC_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'weekend_screen_time']

y = train_df[TARGET].values
print(f'Training shape: {train_df.shape}, Test shape: {test_df.shape}, Target rate: {y.mean():.4f}')

In [ ]:
# 1. Transductive XGBoost Imputation
IMP_PARAMS = dict(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=20, tree_method='hist',
                  device='cuda', enable_categorical=True)

def impute_transductive(tr, te, seed=42):
    n = len(tr)
    full = pd.concat([tr[ALL_RAW], te[ALL_RAW]], ignore_index=True)
    X = full.copy()
    for c in CATS:
        X[c] = X[c].astype('category')
    out = full[NUMS].copy()
    for col in NUMS:
        obs = X[col].notna().values
        feats = [c for c in ALL_RAW if c != col]
        m = xgb.XGBRegressor(**IMP_PARAMS, random_state=seed).fit(X.loc[obs, feats], X.loc[obs, col])
        if (~obs).sum():
            out.loc[~obs, col] = m.predict(X.loc[~obs, feats])
    return out.iloc[:n].reset_index(drop=True), out.iloc[n:].reset_index(drop=True)

tr_imp, te_imp = impute_transductive(train_df, test_df)
print('Transductive imputation completed.')

In [ ]:
# 2. Composition Features & Decimal Lattice
def build_augmented_fe(imp, orig):
    X = imp.copy()
    d, s, g = X.daily_screen_time_hours, X.social_media_hours, X.gaming_hours
    w, wk, sl = X.work_study_hours, X.weekend_screen_time, X.sleep_hours
    n, o = X.notifications_per_day, X.app_opens_per_day
    parts = s + g + w
    
    X['resid'] = d - parts
    X['leisure'] = d - w
    X['social_frac'] = s / (d + 1e-5)
    X['work_frac'] = w / (d + 1e-5)
    X['leisure_frac'] = (d - w) / (d + 1e-5)
    X['resid_frac'] = (d - parts) / (d + 1e-5)
    X['wk_ratio'] = wk / (d + 1e-5)
    X['week_total'] = 5 * d + 2 * wk
    X['awake_screen_frac'] = d / (24.0 - sl + 1e-5)
    X['free_time'] = 24.0 - sl - d - w
    X['notif_per_open'] = n / (o + 1e-5)
    X['min_per_open'] = d * 60.0 / (o + 1e-5)
    
    for c in CATS:
        X[c] = orig[c].astype('category').values
    for c in ALL_RAW:
        X[f'na_{c}'] = orig[c].isna().astype(np.int8).values
    for c in NUMS:
        X[f'rawnan_{c}'] = orig[c].values
        
    return X

X_aug_tr = build_augmented_fe(tr_imp, train_df)
X_aug_te = build_augmented_fe(te_imp, test_df)

def build_lattice(df):
    o = {}
    for c in FRAC_COLS:
        v = df[c].values
        o[f'frac_{c}'] = v - np.floor(v)
        o[f'd1_{c}'] = np.floor(v * 10.0) % 10.0
        o[f'is_int_{c}'] = (v == np.floor(v)).astype(np.float32)
        o[f'is_half_{c}'] = (np.abs(v - np.floor(v) - 0.5) < 1e-4).astype(np.float32)
    return pd.DataFrame(o, index=df.index).astype(np.float32)

LAT_TR = build_lattice(train_df)
LAT_TE = build_lattice(test_df)

def get_levels(df):
    return pd.DataFrame({c: df[c].astype(object).fillna('__missing__').astype(str).values
                         for c in ALL_RAW}, index=df.index)

LTR = get_levels(train_df)
LTE = get_levels(test_df)
print(f'Feature space constructed: {X_aug_tr.shape[1]} augmented + {LAT_TR.shape[1]} lattice features.')

In [ ]:
# 3. Target Encoding Function
ORDER = [f'te_{c}' for c in ALL_RAW] + [f'fq_{c}' for c in ALL_RAW]
SMOOTH = 10.0

full_levels = pd.concat([LTR, LTE], axis=0)
FREQ_MAPS = {c: full_levels[c].value_counts().to_dict() for c in ALL_RAW}

def maps_from(levels, yy):
    gm = yy.mean()
    m = {}
    for c in ALL_RAW:
        g = pd.DataFrame({'lv': levels[c].values, 'y': yy}).groupby('lv')['y'].agg(['count', 'mean'])
        smooth_stat = (g['count'] * g['mean'] + SMOOTH * gm) / (g['count'] + SMOOTH)
        m[c] = smooth_stat.to_dict()
    return m, gm

def apply_maps(levels, m, gm):
    out = {}
    for c in ALL_RAW:
        tmap = m[c]
        out[f'te_{c}'] = levels[c].map(tmap).fillna(gm).values.astype(np.float32)
        out[f'fq_{c}'] = levels[c].map(FREQ_MAPS[c]).fillna(0.0).values.astype(np.float32)
    return pd.DataFrame(out, index=levels.index)[ORDER]

def build_enc(itr, iva):
    y_tr = y[itr]
    L = LTR.iloc[itr].reset_index(drop=True)
    holder = np.zeros((len(itr), len(ORDER)), dtype=np.float32)
    for i_in, i_out in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(itr)), y_tr):
        m, gm = maps_from(L.iloc[i_in], y_tr[i_in])
        holder[i_out] = apply_maps(L.iloc[i_out].reset_index(drop=True), m, gm).values
    m, gm = maps_from(L, y_tr)
    return (pd.DataFrame(holder, columns=ORDER),
            apply_maps(LTR.iloc[iva].reset_index(drop=True), m, gm),
            apply_maps(LTE, m, gm))

print('Bayesian Target Encoder initialized.')

In [ ]:
# 4. Multi-Scale 10-Fold Model Training
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=2026)

PREDS_OOF = {
    'xgb_d6_hist': np.zeros(len(train_df)),
    'xgb_d5_reg': np.zeros(len(train_df)),
    'lgb_num63': np.zeros(len(train_df)),
    'xgb_d8_deep': np.zeros(len(train_df)),
    'cat_native_gpu': np.zeros(len(train_df))
}

PREDS_TEST = {
    'xgb_d6_hist': np.zeros(len(test_df)),
    'xgb_d5_reg': np.zeros(len(test_df)),
    'lgb_num63': np.zeros(len(test_df)),
    'xgb_d8_deep': np.zeros(len(test_df)),
    'cat_native_gpu': np.zeros(len(test_df))
}

num_cols = [c for c in X_aug_tr.columns if not isinstance(X_aug_tr[c].dtype, pd.CategoricalDtype)]
def cat_native_frame(num_block, lvl_block):
    n = num_block.reset_index(drop=True).replace([np.inf, -np.inf], np.nan)
    out = pd.concat([n, lvl_block.reset_index(drop=True).add_prefix('lvl_')], axis=1)
    lvl = [c for c in out.columns if c.startswith('lvl_')]
    for c in lvl:
        out[c] = out[c].astype(str)
    return out, [out.columns.get_loc(c) for c in lvl]

for fold, (itr, iva) in enumerate(skf.split(train_df, y)):
    y_tr, y_va = y[itr], y[iva]
    
    Xa_base = pd.concat([X_aug_tr.iloc[itr].reset_index(drop=True), LAT_TR.iloc[itr].reset_index(drop=True)], axis=1)
    Xb_base = pd.concat([X_aug_tr.iloc[iva].reset_index(drop=True), LAT_TR.iloc[iva].reset_index(drop=True)], axis=1)
    Xt_base = pd.concat([X_aug_te.reset_index(drop=True), LAT_TE.reset_index(drop=True)], axis=1)
    
    e_tr, e_va, e_te = build_enc(itr, iva)
    Xa_te = pd.concat([Xa_base, e_tr], axis=1)
    Xb_te = pd.concat([Xb_base, e_va], axis=1)
    Xt_te = pd.concat([Xt_base, e_te], axis=1)
    
    # 1. XGBoost Depth-6 Hist
    m1 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.028, max_depth=6, min_child_weight=20,
        subsample=0.80, colsample_bytree=0.80, tree_method='hist', device='cuda',
        enable_categorical=True, random_state=2026+fold, early_stopping_rounds=100
    )
    m1.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgb_d6_hist'][iva] = m1.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgb_d6_hist'] += m1.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 2. XGBoost Depth-5 Regularized
    m2 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.032, max_depth=5, min_child_weight=40,
        subsample=0.80, colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=6.0,
        tree_method='hist', device='cuda', enable_categorical=True,
        random_state=3026+fold, early_stopping_rounds=100
    )
    m2.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgb_d5_reg'][iva] = m2.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgb_d5_reg'] += m2.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 3. LightGBM NumLeaves=63
    m3 = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.028, num_leaves=63, max_depth=8,
        colsample_bytree=0.80, subsample=0.80, subsample_freq=1, min_child_samples=80,
        random_state=2026+fold, n_jobs=4, verbose=-1
    )
    m3.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    PREDS_OOF['lgb_num63'][iva] = m3.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['lgb_num63'] += m3.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 4. Deep XGBoost Depth-8 Hist
    m4 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.022, max_depth=8, min_child_weight=35,
        subsample=0.80, colsample_bytree=0.70, reg_alpha=0.20, reg_lambda=2.0,
        tree_method='hist', device='cuda', enable_categorical=True,
        random_state=4026+fold, early_stopping_rounds=100
    )
    m4.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgb_d8_deep'][iva] = m4.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgb_d8_deep'] += m4.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 5. CatBoost Native Ordered TE
    Xa_cat, ci = cat_native_frame(X_aug_tr.iloc[itr][num_cols], LTR.iloc[itr])
    Xb_cat, _  = cat_native_frame(X_aug_tr.iloc[iva][num_cols], LTR.iloc[iva])
    Xt_cat, _  = cat_native_frame(X_aug_te[num_cols], LTE)
    m5 = CatBoostClassifier(
        iterations=1800, learning_rate=0.040, depth=6, eval_metric='AUC',
        early_stopping_rounds=100, task_type='GPU', random_seed=2026+fold, verbose=False
    )
    m5.fit(Xa_cat, y_tr, eval_set=(Xb_cat, y_va), cat_features=ci, verbose=False)
    PREDS_OOF['cat_native_gpu'][iva] = m5.predict_proba(Xb_cat)[:, 1]
    PREDS_TEST['cat_native_gpu'] += m5.predict_proba(Xt_cat)[:, 1] / N_SPLITS
    
    print(f'Fold {fold+1} finished.')
    gc.collect()

print('\nMulti-scale models trained across 10 folds.')

In [ ]:
# 5. Cross-Fitted Logit Meta-Stacker & Rank-01 Percentile Fusion
def to_logit(p, clip=30.0):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1.0 - 1e-15)
    return np.clip(np.log(p / (1.0 - p)), -clip, clip)

def rank01(arr):
    return (rankdata(arr, method='average') - 1.0) / (len(arr) - 1.0)

names = list(PREDS_OOF)
Z_oof = np.column_stack([to_logit(PREDS_OOF[n]) for n in names])
Z_test = np.column_stack([to_logit(PREDS_TEST[n]) for n in names])

stacked_oof = np.zeros(len(y))
stream2_test = np.zeros(len(test_df))

meta_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
for itr, iva in meta_skf.split(Z_oof, y):
    meta = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
    meta.fit(Z_oof[itr], y[itr])
    stacked_oof[iva] = meta.predict_proba(Z_oof[iva])[:, 1]
    stream2_test += meta.predict_proba(Z_test)[:, 1] / 10

# Load Stream 1 Champion
champ_sub = pd.read_csv('submission.csv')
r_champ = rank01(champ_sub['addicted_label'].values)
r_stream2 = rank01(stream2_test)

fused_rank = 0.55 * r_champ + 0.45 * r_stream2
final_predictions = rank01(fused_rank)

sub = pd.DataFrame({'id': test_df['id'].values, TARGET: final_predictions})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv successfully saved: {len(sub)} samples, mean prob: {final_predictions.mean():.6f}')